# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [13]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import numpy as np

## 1.2 Функции

In [15]:
def vectorize_headings_tfidf(df, text_columns=None, max_features=1000, **tfidf_params):
    """
    Векторизует текстовые колонки с помощью TF-IDF
    
    Parameters:
    df - исходный DataFrame
    text_columns - список колонок для обработки
    max_features - максимальное количество фичей TF-IDF
    tfidf_params - дополнительные параметры для TfidfVectorizer
    """
    
    if text_columns is None:
        text_columns = [col for col in df.columns if col.startswith('heading_')]
    
    # Создаем копию DataFrame без текстовых колонок
    df_vectorized = df.drop(columns=text_columns, errors='ignore').copy()
    
    # Параметры TF-IDF по умолчанию
    default_params = {
        'max_features': max_features,
        'min_df': 2,
        'max_df': 0.8,
        'ngram_range': (1, 2),
        'stop_words': None  # т.к. уже предобработали
    }
    default_params.update(tfidf_params)
    
    for col in text_columns:
        if col not in df.columns:
            continue
            
        print(f"Векторизация колонки: {col}")
        
        # Заменяем NaN на пустые строки для TF-IDF
        texts = df[col].fillna('')
        
        # Создаем и обучаем TF-IDF векторизатор
        vectorizer = TfidfVectorizer(**default_params)
        
        try:
            # Векторизуем тексты
            tfidf_matrix = vectorizer.fit_transform(texts)
            
            # Преобразуем в DataFrame с понятными названиями колонок
            feature_names = [f"{col}_tfidf_{name}" for name in vectorizer.get_feature_names_out()]
            tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=feature_names, index=df.index)
            
            # Добавляем к результату
            df_vectorized = pd.concat([df_vectorized, tfidf_df], axis=1)
            
            print(f"Создано {len(feature_names)} признаков для {col}")
            
        except Exception as e:
            print(f"Ошибка при векторизации {col}: {e}")
            # Если ошибка, создаем пустые колонки
            empty_df = pd.DataFrame(index=df.index)
            df_vectorized = pd.concat([df_vectorized, empty_df], axis=1)
    
    return df_vectorized

# 2 Чтение данных

In [18]:
data = pd.read_csv("../../data/news/3_titles_processed.csv")

In [20]:
data.head(2)

,begin,heading_interfax,heading_vedomosti,heading_kommersant
0,2022-05-01 10:00:00,володин предлож конфисковыва актив владельц би...,NaN,росс начина при заявлен нов выплат нужда дет с...
1,2022-05-01 12:00:00,мишустин подписа постановлен снижен ставк льго...,правительств одобр снижен ставк льготн ипотек ...,NaN


# 3 Обработка

In [35]:
df_vectorized = vectorize_headings_tfidf(
    data, 
    max_features=500,  # вместо 10000
    min_df=5,          # слова должны встречаться минимум 5 раз
    max_df=0.7         # игнорировать слишком частые слова
)

Векторизация колонки: heading_interfax
Создано 500 признаков для heading_interfax
Векторизация колонки: heading_vedomosti
Создано 500 признаков для heading_vedomosti
Векторизация колонки: heading_kommersant
Создано 500 признаков для heading_kommersant


In [36]:
df_vectorized.head()

,begin,heading_interfax_tfidf_100,heading_interfax_tfidf_2021,heading_interfax_tfidf_2021 год,heading_interfax_tfidf_2022,heading_interfax_tfidf_2022 год,heading_interfax_tfidf_2023,heading_interfax_tfidf_2023 год,heading_interfax_tfidf_2024,heading_interfax_tfidf_2024 год,...,heading_kommersant_tfidf_чист,heading_kommersant_tfidf_чист прибыл,heading_kommersant_tfidf_шойг,heading_kommersant_tfidf_штраф,heading_kommersant_tfidf_экономик,heading_kommersant_tfidf_экономическ,heading_kommersant_tfidf_экс,heading_kommersant_tfidf_экспорт,heading_kommersant_tfidf_юан,heading_kommersant_tfidf_январ
0,2022-05-01 10:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0
1,2022-05-01 12:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0
2,2022-05-01 14:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0
3,2022-05-01 16:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.15956,0.0,0.0
4,2022-05-01 18:00:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00000,0.0,0.0


In [41]:
df_vectorized.to_csv("../../data/news/3_titles_tfidf_500.csv", index=False)